## Build a Basic Agent

Instantiate the Local LLM of choice. We will be using `qwen2:7b` from ollama

In [6]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen2:7b",
                   temperature=0.7,
                   verbose=True)


Create a simple agent that can answer questions and make tool calls. This agent will use the `get_weather` tool to get the weather of a particular city

In [8]:
from langchain.agents import create_agent

def get_weather(city: str) -> str:
    """Get the weather for a given city."""
    return f"The weather in {city} is sunny with a temperature of 25°C."

agent = create_agent(
        model=model,
        tools=[get_weather],
        system_prompt="You are a helpful assistant that can provide weather information."
        )

# Run the agent
agent.invoke({"messages": [{"role": "user", "content": "What's the weather in New York?"}]})

{'messages': [HumanMessage(content="What's the weather in New York?", additional_kwargs={}, response_metadata={}, id='c9edec92-b199-464e-9755-1a683a94e066'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2:7b', 'created_at': '2025-10-24T03:44:53.3092534Z', 'done': True, 'done_reason': 'stop', 'total_duration': 122788524200, 'load_duration': 53271773300, 'prompt_eval_count': 152, 'prompt_eval_duration': 54173236400, 'eval_count': 26, 'eval_duration': 15246109600, 'model_name': 'qwen2:7b', 'model_provider': 'ollama'}, id='lc_run--8c3aed2b-a3a0-4b27-86ef-0232da1ac86f-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'New York'}, 'id': '49ed6179-1abf-481d-a31a-e44fd4a9fdaf', 'type': 'tool_call'}], usage_metadata={'input_tokens': 152, 'output_tokens': 26, 'total_tokens': 178}),
  ToolMessage(content='The weather in New York is sunny with a temperature of 25°C.', name='get_weather', id='4a859fa3-f583-413e-b1a6-6c1374bb4aef', tool_call_id='49ed6179-1abf

## Build a Real-world Agent

Next, build a practical weather forecasting agent that demonstrates key production concepts:
1. Detailed system prompts for better agent behavior
2. Create tools that integrate with external data
3. Model configuration for consistent responses
4. Structured output for predictable results
5. Conversational memory for chat-like interactions
6. Create and run the agent create a fully functional agent

Let’s walk through each step:

### Define the system prompt

The system prompt defines your agent's role behaviour so keep it actionable and specific

In [13]:
system_prompt = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean wherever they are, use the get_user_location tool to find their location."""

### Create Tools

[Tools](https://docs.langchain.com/oss/python/langchain/tools) let a model interact with external systems by calling functions you define. Tools can depend on [runtime context](https://docs.langchain.com/oss/python/langchain/runtime) and also interact with [agent memory](https://docs.langchain.com/oss/python/langchain/short-term-memory).

Notice below how the `get_user_location` tool uses runtime context:

In [9]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime

@tool
def get_weather_for_location(city: str) -> str:
    """Get the weather for a given city."""
    return f"It is always sunny in {city}"

@dataclass
class Context:
    """Custom runtime context schema."""
    user_id: str

@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """Get the user's location based on their user ID."""
    user_id = runtime.context.user_id
    # In a real application, you would look up the user's location from a database
    return "Florida" if user_id =="1" else "SF"

Tools should be well-documented: their name, description, and argument names become part of the model’s prompt. LangChain’s @tool decorator adds metadata and enables runtime injection via the ToolRuntime parameter.

Configure your model

Set up your language model with the right parameters for your use case:

Since we already have `qwen2:7b` setup we'll skip this

In [ ]:
# Configure your model
"""
from langchain.chat_models import init_chat_model

model = init_chat_model(
        "anthropic:claude-sonnet-4-5",
        temperature=0.5,
        timeout=10,
        max_tokens=1000
    )
"""

### Define response format

Optionally, define a structured response format if you need the agent responses to match a specific schema.

In [10]:
from dataclasses import dataclass
# We use a dataclass here but you can also use other schema definitions like Pydantic models

@dataclass
class Responseformat:
    """Response format schema for the agent."""
    # A puunny response about the weather is required
    punny_repsonse: str
    # Any other useful information about the weather if available
    weather_conditions: str | None = None

### Add memory

Add [memory](https://docs.langchain.com/oss/python/langchain/short-term-memory) to your agent to maintain state across interactions. This allows the agent to remember previous conversations and context.

In [11]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

### Create and run the agent

Now assemble your agent with all the components and run it!

In [15]:
agent = create_agent(
        model=model,
        tools=[get_weather_for_location, get_user_location],
        system_prompt=system_prompt,
        context_schema=Context,
        response_format=Responseformat,
        checkpointer=checkpointer
)

# 'thread_id' is a unique identifier for the conversation thread
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather outside?"}]},
    context=Context(user_id="1"),
    config=config
)

print(response)

# ResponseFormat(
#     punny_response="Florida is still having a 'sun-derful' day! The sunshine is playing 'ray-dio' hits all day long! I'd say it's the perfect weather for some 'solar-bration'! If you were hoping for rain, I'm afraid that idea is all 'washed up' - the forecast remains 'clear-ly' brilliant!",
#     weather_conditions="It's always sunny in Florida!"
# )

# Note that we can continue the conversation using the same thread_id to maintain context
response = agent.invoke(
    {"messages": [{"role": "user", "content": "What about in New York?"}]},
    context=Context(user_id="1"),
    config=config
)

print(response)
# ResponseFormat(
#     punny_response="Florida is still having a 'sun-derful' day! The sunshine is playing 'ray-dio' hits all day long! I'd say it's the perfect weather for some 'solar-bration'! If you were hoping for rain, I'm afraid that idea is all 'washed up' - the forecast remains 'clear-ly' brilliant!",
#     weather_conditions="It's always sunny in Florida!"
# )

{'messages': [HumanMessage(content='What is the weather outside?', additional_kwargs={}, response_metadata={}, id='401f46e2-2bef-4aea-83db-58b0e11f97db'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2:7b', 'created_at': '2025-10-24T04:27:16.5716702Z', 'done': True, 'done_reason': 'stop', 'total_duration': 142275843100, 'load_duration': 27167729000, 'prompt_eval_count': 343, 'prompt_eval_duration': 102696891800, 'eval_count': 22, 'eval_duration': 12286935700, 'model_name': 'qwen2:7b', 'model_provider': 'ollama'}, id='lc_run--bcbb0144-f2c6-4e06-b9c5-0c80b84781e6-0', tool_calls=[{'name': 'get_user_location', 'args': {}, 'id': '187efc36-25b9-4e45-922a-65e12c4e0cb8', 'type': 'tool_call'}], usage_metadata={'input_tokens': 343, 'output_tokens': 22, 'total_tokens': 365}), ToolMessage(content='Florida', name='get_user_location', id='bf29e86f-24bf-405e-9693-fbdb49d10be9', tool_call_id='187efc36-25b9-4e45-922a-65e12c4e0cb8'), AIMessage(content='', additional_kwarg